# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# My Rule and Its Reason Codes

## Rule

Pages with high impressions, low CTR, stale content, and declining traffic should be prioritized for content refresh because they have the greatest opportunity for SEO improvement.

## Reason Codes

- LOW_CTR
- HIGH_IMPRESSIONS
- STALE_CONTENT
- DECLINING_TREND
- REFRESH_PRIORITY

This baseline scoring rule is intended as decision-support and uses only observed historical signals.

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("../outputs/refresh_queue_sample.csv")

print(df.head())
print(df.shape)

   final_rank            content_id          client_id  final_refresh_score  \
0           1  content_1f080331fa2b  client_3fdba35f04            81.636697   
1           2  content_6aa43079fb0c  client_3fdba35f04            81.447656   
2           3  content_d6570c51c9bd  client_3fdba35f04            81.430346   
3           4  content_72e800a9c214  client_3fdba35f04            81.034960   
4           5  content_e04eb9549989  client_3fdba35f04            80.873188   

  best_model_name  best_model_probability  baseline_refresh_score confidence  \
0   random_forest                0.782079                0.844481       high   
1   random_forest                0.788105                0.825477       high   
2   random_forest                0.847372                0.695884     medium   
3   random_forest                0.774371                0.842545       high   
4   random_forest                0.814805                0.749468     medium   

         suggested_action                   

## Signal Check 1

CTR vs Suggested Action

Goal:
Determine whether pages with lower CTR receive higher refresh priority.

Verdict:
CONFIRMED

In [22]:
ctr_table = df.groupby("suggested_action")["ctr"].agg(["count","mean"])

print(ctr_table)

                               count      mean
suggested_action                              
refresh                           35  0.081143
refresh_and_review_ctr           130  0.134077
refresh_and_review_engagement     35  0.318571


## Signal Check 2

Content Age vs Refresh Score

Goal:
Determine whether older pages receive higher baseline refresh scores.

Verdict:
CONFIRMED

In [23]:
age_table = df.groupby("freshness_tier")["baseline_refresh_score"].agg(["count","mean"])

print(age_table)

                count      mean
freshness_tier                 
0-30               65  0.640490
181+                1  0.824268
91-180            134  0.708332


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

# Build Ranked Queue

A simple baseline score is created using impressions, CTR, and content age.

Higher score indicates higher refresh priority.

In [24]:
df["baseline_score"] = (
    df["impressions_90d"] / 1000
    - df["ctr"]
    + df["content_age_days"] / 100
)

df["reason_code"] = "REFRESH_PRIORITY"

df["action"] = df["suggested_action"]

ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

ranked.head()

,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,baseline_score,reason_code,action
99,100,content_9d9905bcb297,client_7f2253d7e2,77.578361,random_forest,0.795638,0.688710,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,keyword article,transactional,91-180,0-30,3500+,excellent,striking,46.407,REFRESH_PRIORITY,refresh_and_review_ctr
130,131,content_2333ccd359f7,client_7f2253d7e2,77.339264,random_forest,0.813368,0.642664,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,keyword article,commercial,181-365,0-30,3500+,excellent,page_3_5,45.380,REFRESH_PRIORITY,refresh_and_review_engagement
189,190,content_454e62c347a0,client_7f2253d7e2,76.899128,random_forest,0.766487,0.731057,high,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,...,keyword article,informational,181-365,0-30,3500+,excellent,page_1,42.414,REFRESH_PRIORITY,refresh_and_review_ctr
31,32,content_964dd0100c99,client_7f2253d7e2,78.945092,random_forest,0.839227,0.636309,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,keyword article,informational,91-180,0-30,3500+,excellent,page_3_5,42.398,REFRESH_PRIORITY,refresh_and_review_engagement
134,135,content_341b1ecdea85,client_7f2253d7e2,77.318512,random_forest,0.820299,0.626925,high,refresh_and_review_engagement,declining_with_demand|low_engagement_visible_p...,...,keyword article,informational,181-365,0-30,3500+,excellent,page_3_5,42.036,REFRESH_PRIORITY,refresh_and_review_engagement


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# Top-20 Review

Each row includes:

- Action
- Reason Code
- Confidence
- What would make it wrong

In [26]:
top20 = ranked.head(20)

for _, row in top20.iterrows():

    print(f"""
Content : {row['content_id']}
Action : {row['action']}
Reason : {row['reason_code']}
Confidence : {row['confidence']}
What would make it wrong :
Recent improvements or updated content not yet reflected in the dataset.
""")


Content : content_9d9905bcb297
Action : refresh_and_review_ctr
Reason : REFRESH_PRIORITY
Confidence : high
What would make it wrong :
Recent improvements or updated content not yet reflected in the dataset.


Content : content_2333ccd359f7
Action : refresh_and_review_engagement
Reason : REFRESH_PRIORITY
Confidence : high
What would make it wrong :
Recent improvements or updated content not yet reflected in the dataset.


Content : content_454e62c347a0
Action : refresh_and_review_ctr
Reason : REFRESH_PRIORITY
Confidence : high
What would make it wrong :
Recent improvements or updated content not yet reflected in the dataset.


Content : content_964dd0100c99
Action : refresh_and_review_engagement
Reason : REFRESH_PRIORITY
Confidence : high
What would make it wrong :
Recent improvements or updated content not yet reflected in the dataset.


Content : content_341b1ecdea85
Action : refresh_and_review_engagement
Reason : REFRESH_PRIORITY
Confidence : high
What would make it wrong :
Recent i

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak Picks

Some lower-ranked pages may already perform well despite moderate scores.

These pages should be reviewed manually before taking action.

## Leakage Check

No future performance information was used.

No labels or product flags were used for scoring.

The rule relies only on observed historical features.

In [27]:
print("Lowest Ranked Pages")

ranked.tail(5)[
    [
        "content_id",
        "baseline_score",
        "action"
    ]
]

Lowest Ranked Pages


,content_id,baseline_score,action
185,content_412512ea6fc3,2.316,refresh_and_review_ctr
166,content_d4c50bd49b5d,2.275,refresh_and_review_ctr
76,content_5cda32af644c,2.200,refresh_and_review_ctr
79,content_b005f46f2e4c,2.173,refresh_and_review_ctr
97,content_b893db8d7619,2.058,refresh_and_review_ctr


In [29]:
print("Leakage Check")

print("✓ No future window used")

print("✓ No label-derived features used")

print("✓ No product flags used")

Leakage Check
✓ No future window used
✓ No label-derived features used
✓ No product flags used


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.